# Data Challenge 13 — Interpreting Logistic Regression 

**Purpose**  
Apply what you learned about logistic regression interpretation by analyzing NYC Restaurant Inspection data. 
 
You’ll practice interpreting **continuous**, **binary**, and **categorical** predictors, compute **odds ratios**, and assess model accuracy. 

**Learning Goals**
- Convert coefficients to odds ratios using `np.exp()`.  
- Interpret ORs for continuous, binary, and categorical predictors.  
- Use accuracy to assess logistic regression performance.  
- Communicate results clearly and responsibly.  

**Data:** June 1, 2025 - Nov 4, 2025 Restaurant Health Inspection

[Restaurant Health Inspection](https://data.cityofnewyork.us/Health/DOHMH-New-York-City-Restaurant-Inspection-Results/43nn-pn8j/about_data)


## Instructor Guidance

**Hint: Use the Lecture Deck, Canvas Reading, and Docs to help you with the code**

Use this guide live; students implement below.

**Docs (Quick Links)**
- LogisticRegression — https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html  
- accuracy_score — https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html  
- OneHotEncoder — https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html  
- StandardScaler — https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html  
- np.exp — https://numpy.org/doc/stable/reference/generated/numpy.exp.html  

**Pseudocode Plan**

1️⃣ Load cleaned restaurant inspection data from the previous challenge.  
2️⃣ Define target = `IS_A` (1 = Grade A, 0 = otherwise).  
3️⃣ Predictors →  
    • Continuous = `SCORE`  
    • Binary = `CRITICAL_NUM`  
    • Categorical = `BORO`  
4️⃣ Scale continuous variables; encode categorical ones.  
5️⃣ Fit `LogisticRegression`.  
6️⃣ Exponentiate coefficients (np.exp()) → odds ratios.  
7️⃣ Interpret one continuous, one binary, and one categorical coefficient.  
8️⃣ Evaluate accuracy.  
9️⃣ Reflect on scaling choices and communication of odds.  


## You Do — Student Section
Work in pairs. Comment your choices briefly. Keep code simple—only coerce the columns you use.

## Step 1 — Imports and Plot Defaults

In [71]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression,LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.metrics import mean_squared_error,accuracy_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
import seaborn as sns
from sklearn import preprocessing


### Step 2 — Load CSV, Create Columns, Preview

- Point to your New York City Restaurant Inspection Data 
- Create the `is_A` and `critical_num` columns like you did in L11 notebook

In [72]:
df = pd.read_csv("../data/DOHMH_New_York_City_Restaurant_Inspection_Results_20251104 copy.csv")

In [73]:
#turning SCORE column to numeric
df['SCORE'] = pd.to_numeric(df['SCORE'])

# creating new df where score isn't NA
df_wscore = df[df['SCORE'].notna()]

# boolean to determine if grade is 'A'
df_wscore['IS_A'] = (df['GRADE'] == 'A').astype(int)

# boolean to determine if Critical flag is 'Critical'
df_wscore['CRITICAL_NUM'] = (df_wscore['CRITICAL FLAG'] == 'Critical').astype(int)

display(df_wscore.shape)
display(df_wscore.info())

/var/folders/3y/ldxff8k17wjcyzkq19xw1srm0000gn/T/ipykernel_21049/2019678538.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_wscore['IS_A'] = (df['GRADE'] == 'A').astype(int)
/var/folders/3y/ldxff8k17wjcyzkq19xw1srm0000gn/T/ipykernel_21049/2019678538.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_wscore['CRITICAL_NUM'] = (df_wscore['CRITICAL FLAG'] == 'Critical').astype(int)


(274939, 29)

<class 'pandas.core.frame.DataFrame'>
Index: 274939 entries, 18 to 291277
Data columns (total 29 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   CAMIS                  274939 non-null  int64  
 1   DBA                    274939 non-null  object 
 2   BORO                   274939 non-null  object 
 3   BUILDING               274150 non-null  object 
 4   STREET                 274939 non-null  object 
 5   ZIPCODE                272033 non-null  float64
 6   PHONE                  274934 non-null  object 
 7   CUISINE DESCRIPTION    274939 non-null  object 
 8   INSPECTION DATE        274939 non-null  object 
 9   ACTION                 274939 non-null  object 
 10  VIOLATION CODE         273397 non-null  object 
 11  VIOLATION DESCRIPTION  273397 non-null  object 
 12  CRITICAL FLAG          274939 non-null  object 
 13  SCORE                  274939 non-null  float64
 14  GRADE                  142194 non-null  

None

## Step 3 — Define Predictors & Target

- Target is `is_A` 
- X predictors are: SCORE, CRITICAL_NUM (created in Step 2), BORO


In [74]:
#dummies_pd = (pd.get_dummies(df_wscore['BORO'], drop_first=False, prefix='borough')).astype(int)
#df_with_pd = pd.concat([df_wscore, dummies_pd], axis=1)

testdf = df_wscore[['IS_A','SCORE','CRITICAL_NUM','BORO']]

y = testdf['IS_A']
X = testdf[['SCORE','CRITICAL_NUM','BORO']]

## Step 4 — Split Data (70/30 Stratify by Target)

In [75]:
np.random.seed(16)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=16)


## Step 5 – Preprocessing (You can chose to do this in a Pipeline)  

- Scale continuous features  
- Pass binary as is  
- One-hot encode categorical feature (`BORO`)  

In [80]:
# defining preprocessing for columns that are being transformed
preprocessor = ColumnTransformer(
        transformers=[
            # transformed for numeric column
            ('numeric',StandardScaler(),['SCORE']),
            # transforming for categorical columns
            ('categorical', OneHotEncoder(handle_unknown='ignore'),['BORO'])
        ],
        # ignoring columns that were not referenced in preprocessing
        remainder='passthrough'
)

# defining pipeline
model = Pipeline(
    steps=
        [
        # using predefined processing for model
        ('processing',preprocessor),
        # regression that would be performed
        ('model', LogisticRegression())
        ]
)


## Step 6 – Fit Model & Evaluate Accuracy

- Fit `is_A ~ score` using **LogisticRegression**  
- Compute predictions with `.predict()`  
- Evaluate accuracy with `accuracy_score()`

In [82]:
model.fit(X_train,y_train)

predictions = model.predict(X_test)

print('The Accuracy Score is:',round(accuracy_score(y_test, (predictions >= 0.5).astype(int)),5))

The Accuracy Score is: 0.95137


## Step 7 – Extract Coefficients and Convert to Odds Ratios


In [83]:
names = preprocessor.get_feature_names_out()
coefficients = model.named_steps['model'].coef_[0]
oddsratio = np.exp(coefficients)

info = pd.DataFrame({
    'columns':names,
    'coefficients':coefficients,
    'oddsratio':oddsratio
})

info

,columns,coefficients,oddsratio
0,numeric__SCORE,-6.075310,0.002299
1,categorical__BORO_Bronx,-0.597135,0.550386
2,categorical__BORO_Brooklyn,-0.565791,0.567911
3,categorical__BORO_Manhattan,-0.545968,0.579281
4,categorical__BORO_Queens,-0.466270,0.627338
5,categorical__BORO_Staten Island,-0.517743,0.595864
6,remainder__CRITICAL_NUM,-0.087536,0.916186


## Step 8 – Interpret Each Predictor 

**Remember**
💡 OR > 1 → increases odds of Grade A  
💡 OR < 1 → decreases odds of Grade A

**Type markdown interpreting all 3 predictors in plain english**


# We Share — Reflection & Wrap-Up

Write **one short paragraphs** (4–6 sentences). Be specific and use evidence from your notebook.

**Which predictor had the strongest relationship with getting an A grade?**  
Use the odds ratios and accuracy to support your answer.  